# 도쿄 청과 냉장센터 — 입하량 예측 · 통계 · 입고 예정 (Python 파트)

`mikang_정리.ipynb`의 특징 10개 + 랜덤포레스트 모델을 귤·사과·양파 3품목으로 확장해서, **ERP 입고 예정**을 만든다.

```
[이 노트북 / forecast.py]  시장 데이터 → 통계·그래프 → 랜덤포레스트 예측 → 입고 예정 (상자)
          │  POST /internal/inbound/plans
          ▼
[Spring ERP]  입고 예정 대비 검증 (±10%, 예외 입고, 창고 82팔레트) · 승인 · Oracle
          ▲
[ai_service.py]  Ollama Tool Calling 에이전트 + Qdrant 벡터 검색(RAG)
```

같은 폴더에 `forecast.py`와 CSV(`東京_みかん_日別_*.csv`, `東京_みかん_産地別_*.csv` 등)를 둔다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

import forecast as fc

# ERP 연결 (입고 예정 저장할 때만 필요)
ERP_URL = "http://ERP_SERVER_IP:8090"
ERP_TOKEN = ""            # Spring erp.internal.token 과 같은 값
AI_URL = "http://127.0.0.1:8000"   # ai_service.py (uvicorn) 주소
PUSH = False              # True 로 바꾸면 예측한 입고 예정을 ERP에 저장

ITEMS = fc.available_items()
print("찾은 품목:", ITEMS)
for item in ITEMS:
    print(f"  {item}: {fc.find_file(item, '日別')} / 산지별: {fc.find_file(item, '産地別')}")

## 1. 데이터 개요 · 통계

In [ ]:
data = {item: fc.load_daily(item) for item in ITEMS}

rows = []
for item, df in data.items():
    op = df[df["market_open"] & (df["qty_kg"] > 0)]
    rows.append({
        "품목": item,
        "기간": f"{df['date'].min().date()} ~ {df['date'].max().date()}",
        "입하 영업일": len(op),
        "일평균(톤)": round(op["qty_kg"].mean() / 1000, 1),
        "중앙값(톤)": round(op["qty_kg"].median() / 1000, 1),
        "최대(톤)": round(op["qty_kg"].max() / 1000, 1),
        "변동계수": round(op["qty_kg"].std() / op["qty_kg"].mean(), 2),
        "평균가격(엔/kg)": round(op["price_yen_per_kg"].mean()),
    })
summary = pd.DataFrame(rows).set_index("품목")
summary

In [ ]:
fig, axes = plt.subplots(len(ITEMS), 1, figsize=(14, 3.2 * len(ITEMS)), sharex=True)
axes = np.atleast_1d(axes)
for ax, item in zip(axes, ITEMS):
    op = data[item][data[item]["market_open"]]
    ax.plot(op["date"], op["qty_kg"] / 1000, linewidth=0.7)
    ax.plot(op["date"], (op["qty_kg"] / 1000).rolling(10, min_periods=1).mean(), linewidth=1.8, label="10-day avg")
    ax.set_title(f"Tokyo {fc.ITEMS[item]['en']} - Daily Arrival (tons)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**보관성에 따라 입하 패턴이 다르다**: 귤은 성수기(9~3월)에만 몰리고 연말에 치솟는다. 사과·양파는 산지 저장고가 물량을 풀어서 1년 내내 들어온다.

In [ ]:
fig, axes = plt.subplots(1, len(ITEMS), figsize=(5 * len(ITEMS), 4))
axes = np.atleast_1d(axes)
for ax, item in zip(axes, ITEMS):
    op = data[item][data[item]["market_open"] & (data[item]["qty_kg"] > 0)].copy()
    op["month"] = op["date"].dt.month
    m = op.groupby("month")["qty_kg"].mean() / 1000
    m = m.reindex(range(1, 13), fill_value=0)
    ax.bar(m.index, m.values, color="tab:orange")
    ax.set_title(f"{fc.ITEMS[item]['en']} - Avg tons/day by Month")
    ax.set_xticks(range(1, 13))
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 요일별 · 휴장 다음 날 효과 (노트북 3장과 같은 분석)
wd = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
tables = []
for item in ITEMS:
    f = fc.make_features(data[item])
    f = f[f["month"].isin(fc.ITEMS[item]["months"]) & (f["qty_kg"] > 0)]
    t = (f.groupby("weekday_num")["qty_kg"].mean() / 1000).round(1).rename(index=wd)
    t.name = item
    tables.append(t)
    up = f.groupby("after_closed")["qty_kg"].mean()
    if 1 in up.index and 0 in up.index:
        print(f"{item}: 휴장 다음 날 입하량 = 평소의 {up[1] / up[0]:.2f}배")
print()
print("요일별 평균 입하량 (톤)")
display(pd.concat(tables, axis=1))

## 2. 예측 모델 검증 (최근 60영업일을 그 이전 데이터로만 학습해서 맞히기)

In [ ]:
results, backtests, importances = [], {}, {}
for item in ITEMS:
    bt, s, imp = fc.backtest(item, last_n=60)
    backtests[item], importances[item] = bt, imp
    results.append(s)
score = pd.DataFrame(results).set_index("item")
score.columns = ["검증 일수", "평균오차(톤)", "모델 오차율 WAPE(%)", "기준선(지난주 같은 요일) WAPE(%)"]
score

In [ ]:
fig, axes = plt.subplots(len(ITEMS), 1, figsize=(14, 3.2 * len(ITEMS)))
axes = np.atleast_1d(axes)
for ax, item in zip(axes, ITEMS):
    bt = backtests[item]
    ax.plot(bt["date"], bt["qty_kg"] / 1000, marker=".", label="actual")
    ax.plot(bt["date"], bt["pred_kg"] / 1000, label="random forest")
    ax.plot(bt["date"], bt["last_week_qty"] / 1000, alpha=0.5, label="last week same day")
    ax.set_title(f"{fc.ITEMS[item]['en']} - Backtest (tons)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
imp = pd.DataFrame(importances).fillna(0)
imp.plot.barh(figsize=(9, 5))
plt.title("Feature Importance (Random Forest)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 앞으로 2주 예측 → 우리 창고 입고 예정

- 시장 입하 예측 × 점유율 0.5% × 작년 같은 달 산지 비중 ÷ 상자당 kg = **산지별 입고 예정 상자 수**
- 휴장일(수·일·공휴일)은 예정 없음

In [ ]:
START = pd.Timestamp.today().normalize()
END = START + pd.Timedelta(days=14)

fig, ax = plt.subplots(figsize=(14, 4))
for item in ITEMS:
    f = fc.forecast(item, END)
    f = f[f["date"] >= START - pd.Timedelta(days=14)]
    ax.plot(f["date"], f["pred_kg"] / 1000, marker="o", label=fc.ITEMS[item]["en"])
ax.axvline(START, color="gray", linestyle="--")
ax.set_title("Tokyo Market Arrival Forecast (tons/day)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# ERP 상품 마스터 (ERP에 연결되면 ERP 값을 쓰고, 아니면 아래 기본값)
DEFAULT_PRODUCTS = [
    {"code": "MK-EHM", "category": "귤", "origin": "愛媛", "kg_per_box": 10},
    {"code": "MK-KMM", "category": "귤", "origin": "熊本", "kg_per_box": 10},
    {"code": "MK-NGS", "category": "귤", "origin": "長崎", "kg_per_box": 10},
    {"code": "MK-WKY", "category": "귤", "origin": "和歌山", "kg_per_box": 10},
    {"code": "AP-AOM", "category": "사과", "origin": "青森", "kg_per_box": 10},
    {"code": "AP-NGN", "category": "사과", "origin": "長野", "kg_per_box": 10},
    {"code": "AP-YMG", "category": "사과", "origin": "山形", "kg_per_box": 10},
    {"code": "AP-IWT", "category": "사과", "origin": "岩手", "kg_per_box": 10},
    {"code": "ON-HKD", "category": "양파", "origin": "北海道", "kg_per_box": 20},
]
try:
    products = requests.get(f"{ERP_URL}/internal/products", headers={"X-Internal-Token": ERP_TOKEN}, timeout=5).json()
    assert isinstance(products, list) and products
    print("ERP 상품 마스터 사용:", len(products), "개")
except Exception as e:
    products = DEFAULT_PRODUCTS
    print("ERP 연결 안 됨 → 기본 상품 목록 사용")

plans, detail = fc.build_plans(products, START, END)
table = detail.pivot_table(index="date", columns="code", values="boxes", aggfunc="sum").fillna(0).astype(int)
table["합계(상자)"] = table.sum(axis=1)
table

In [ ]:
# 창고 체크: 하루 입고량이 팔레트 몇 장인지 (10kg 상자 54개 = 540kg = T11 1장)
PALLET_KG = 540
SLOTS = 82
detail["kg"] = detail["boxes"] * detail["code"].map({p["code"]: p["kg_per_box"] for p in products})
daily = detail.groupby("date")["kg"].sum().to_frame()
daily["입고 팔레트"] = np.ceil(daily["kg"] / PALLET_KG).astype(int)
daily["10톤 트럭"] = np.ceil(daily["kg"] / 10000).astype(int)
daily["창고 대비(%)"] = (daily["입고 팔레트"] / SLOTS * 100).round(0)
daily

In [ ]:
if PUSH:
    r = requests.post(f"{ERP_URL}/internal/inbound/plans", json=plans,
                      headers={"X-Internal-Token": ERP_TOKEN}, timeout=20)
    print(r.status_code, r.text)
else:
    print(f"입고 예정 {len(plans)}건 준비됨. ERP에 저장하려면 PUSH = True 로 바꾸고 다시 실행")

## 4. AI 서비스 확인 (ai_service.py 가 떠 있을 때)

- RAG: 매뉴얼을 문단 단위로 청킹 → bge-m3 임베딩 → Qdrant 저장 → **권한(roles) 메타데이터로 필터링**해서 검색
- 에이전트: LLM이 도구를 고르고, 도구는 Spring 내부 API를 호출 (LLM은 DB를 직접 건드리지 않음)

In [ ]:
def rag(query, role="STAFF"):
    r = requests.post(f"{AI_URL}/rag/search", params={"query": query, "role": role}, timeout=60)
    return pd.DataFrame(r.json())

print("STAFF 권한으로 '실사 차이 조정' 검색 → 관리자 문서는 안 나와야 함")
display(rag("재고 실사에서 차이가 나면 어떻게 조정해?", "STAFF"))
print("MANAGER 권한으로 같은 검색")
display(rag("재고 실사에서 차이가 나면 어떻게 조정해?", "MANAGER"))

In [ ]:
r = requests.post(f"{AI_URL}/agent/chat", json={
    "message": "다음 주 귤 시장 입하량 예측 알려줘",
    "role": "STAFF", "actor": "생수", "history": []}, timeout=240)
res = r.json()
print(res["reply"])
print()
print("사용한 도구:", res["toolLog"])